# EDA — FIFA / EA Sports Player Ratings (SofIFA)
**Source**: `sofifa/` — FIFA 22 complete player dataset (stefanoleone992 via Kaggle)

**Purpose**: Profile EA Sports overall ratings as a squad quality proxy. Check position coverage, rating distributions, nationality coverage for WC 2026, and suitability for position-weighted squad aggregation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

sns.set_theme(style='whitegrid', palette='muted')
DATA = Path('../data/raw/sofifa')

print('Files in sofifa dir:')
for f in sorted(os.listdir(DATA)):
    size = os.path.getsize(DATA / f)
    print(f'  {f} ({size/1024:.0f} KB)')

In [ ]:
# Load whichever CSV files are available
csv_files = [f for f in os.listdir(DATA) if f.endswith('.csv')]

if not csv_files:
    print('No CSV files found — SofIFA dataset may not have downloaded correctly.')
    print('Manual download required: go to https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset')
    print('and place the CSV files in data/raw/sofifa/')
else:
    print('Found CSV files:', csv_files)
    # Load the main players file (likely players_22.csv or similar)
    main_file = sorted([f for f in csv_files if 'player' in f.lower() and '22' in f], key=lambda x: x)
    if not main_file:
        main_file = csv_files
    
    dfs = {}
    for f in csv_files:
        try:
            df = pd.read_csv(DATA / f, low_memory=False)
            dfs[f] = df
            print(f'{f}: {df.shape}')
        except Exception as e:
            print(f'{f}: ERROR - {e}')

In [ ]:
if not dfs:
    print('Skipping analysis — no data loaded.')
else:
    # Use the largest file as main player dataset
    main_df_name = max(dfs, key=lambda k: len(dfs[k]))
    players = dfs[main_df_name]
    print(f'Using {main_df_name} as main dataset: {players.shape}')
    print('Columns:', players.columns.tolist()[:30], '...')
    players.head(3)

## 2. Overall Rating Distribution

In [ ]:
if dfs:
    # Find overall rating column
    overall_col = next((c for c in players.columns if 'overall' in c.lower()), None)
    potential_col = next((c for c in players.columns if 'potential' in c.lower()), None)
    nationality_col = next((c for c in players.columns if 'nation' in c.lower()), None)
    position_col = next((c for c in players.columns if c.lower() in ['club_position', 'player_positions', 'position']), None)

    print(f'Overall col: {overall_col}')
    print(f'Potential col: {potential_col}')
    print(f'Nationality col: {nationality_col}')
    print(f'Position col: {position_col}')

    if overall_col:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        players[overall_col].plot(kind='hist', bins=40, ax=axes[0], color='steelblue', edgecolor='white')
        axes[0].set_title('Player Overall Rating Distribution')
        axes[0].set_xlabel('Overall Rating')

        if potential_col:
            (players[potential_col] - players[overall_col]).plot(
                kind='hist', bins=40, ax=axes[1], color='coral', edgecolor='white'
            )
            axes[1].set_title('Potential - Overall (Growth Headroom)')
            axes[1].set_xlabel('Rating Gap')

        plt.tight_layout()
        plt.show()

        print(f'Overall rating range: {players[overall_col].min()} - {players[overall_col].max()}')
        print(f'Overall rating mean: {players[overall_col].mean():.1f}')
        print(f'Players rated 80+: {(players[overall_col] >= 80).sum()}')
        print(f'Players rated 85+: {(players[overall_col] >= 85).sum()}')
        print(f'Players rated 90+: {(players[overall_col] >= 90).sum()}')

## 3. Ratings by Position

In [ ]:
if dfs and overall_col and position_col:
    pos_ratings = players.groupby(position_col)[overall_col].agg(['mean', 'median', 'count'])
    pos_ratings = pos_ratings[pos_ratings['count'] > 50].sort_values('median', ascending=False)
    print('Overall rating by position:')
    print(pos_ratings.round(1))

    fig, ax = plt.subplots(figsize=(12, 5))
    pos_ratings['median'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Median Overall Rating by Position')
    ax.set_xlabel('Median Overall Rating')
    plt.tight_layout()
    plt.show()

## 4. WC 2026 Nations Coverage

In [ ]:
if dfs and overall_col and nationality_col:
    wc2026_teams = [
        'Argentina', 'Brazil', 'France', 'England', 'Spain', 'Germany', 'Portugal', 'Netherlands',
        'Belgium', 'Italy', 'Croatia', 'Switzerland', 'Denmark', 'Mexico', 'United States', 'Canada',
        'Uruguay', 'Colombia', 'Ecuador', 'Chile', 'Paraguay', 'Venezuela', 'Peru', 'Bolivia',
        'Morocco', 'Senegal', 'Nigeria', 'Egypt', 'South Africa', 'Cameroon', "Ivory Coast", 'Ghana',
        'Algeria', 'Tunisia', 'DR Congo', 'Saudi Arabia', 'Japan', 'South Korea', 'Iran', 'Australia',
        'Qatar', 'New Zealand', 'Costa Rica', 'Honduras', 'Panama', 'Jamaica', 'Trinidad and Tobago',
        'Turkey'
    ]

    nation_coverage = players.groupby(nationality_col)[overall_col].agg(
        n_players='count',
        mean_overall='mean',
        top11_mean=lambda x: x.nlargest(11).mean(),
        top18_mean=lambda x: x.nlargest(18).mean()
    ).reset_index()

    wc_coverage = nation_coverage[nation_coverage[nationality_col].isin(wc2026_teams)]
    missing = [t for t in wc2026_teams if t not in nation_coverage[nationality_col].values]

    print(f'WC 2026 nations with coverage: {len(wc_coverage)} / {len(wc2026_teams)}')
    if missing:
        print(f'Missing (name mismatch likely): {missing}')

    print('\nTop 20 WC nations by top-11 mean overall:')
    print(wc_coverage.nlargest(20, 'top11_mean')[
        [nationality_col, 'n_players', 'top11_mean', 'top18_mean']
    ].round(1).to_string(index=False))

In [ ]:
if dfs and overall_col and nationality_col:
    if len(wc_coverage) > 0:
        fig, ax = plt.subplots(figsize=(12, 10))
        plot_data = wc_coverage.sort_values('top18_mean')
        ax.barh(plot_data[nationality_col], plot_data['top18_mean'], color='steelblue')
        ax.axvline(plot_data['top18_mean'].mean(), color='red', linestyle='--', label='Mean')
        ax.set_title('WC 2026 Nations — Top-18 Mean Overall Rating')
        ax.set_xlabel('Mean Overall Rating (top 18 players)')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 5. Position-Weighted Squad Aggregation (Yeung et al. 2023)

In [ ]:
if dfs and overall_col and nationality_col and position_col:
    # Simplified position classification
    def classify_pos(pos_str):
        if pd.isna(pos_str):
            return 'Unknown'
        pos = str(pos_str).upper()
        if any(x in pos for x in ['GK']):
            return 'GK'
        if any(x in pos for x in ['CB', 'LB', 'RB', 'LWB', 'RWB', 'DEF']):
            return 'DEF'
        if any(x in pos for x in ['CDM', 'CM', 'CAM', 'LM', 'RM', 'MID']):
            return 'MID'
        if any(x in pos for x in ['LW', 'RW', 'CF', 'ST', 'ATT', 'FWD']):
            return 'FWD'
        return 'Unknown'

    players_pos = players.copy()
    players_pos['pos_group'] = players_pos[position_col].apply(classify_pos)

    print('Position group distribution:')
    print(players_pos['pos_group'].value_counts())

    # Compute position-weighted squad score per nationality
    # Weights: GK=0, DEF=35%, MID=35%, FWD=30% (Yeung et al. 2023)
    weights = {'DEF': 0.35, 'MID': 0.35, 'FWD': 0.30}

    squad_scores = []
    for nat in wc2026_teams:
        nat_players = players_pos[players_pos[nationality_col] == nat]
        if len(nat_players) == 0:
            continue
        score = 0
        for pos, weight in weights.items():
            pos_players = nat_players[nat_players['pos_group'] == pos][overall_col]
            if len(pos_players) > 0:
                top_n = pos_players.nlargest(4)  # ~4 per position in top 11
                score += weight * top_n.mean()
        squad_scores.append({'nationality': nat, 'weighted_squad_score': score})

    squad_df = pd.DataFrame(squad_scores).sort_values('weighted_squad_score', ascending=False)
    print('\nPosition-weighted squad scores (top 20):')
    print(squad_df.head(20).round(2).to_string(index=False))

## 6. Red Flags Summary

In [ ]:
red_flags = []

if not dfs:
    red_flags.append('SofIFA dataset not downloaded — manual download required from Kaggle')
elif dfs:
    if not overall_col:
        red_flags.append('Could not find overall rating column — check column names in loaded file')
    if not nationality_col:
        red_flags.append('Could not find nationality column — nationality-level aggregation not possible')
    if missing:
        red_flags.append(f'Name mismatches for WC 2026 teams: {missing[:5]} — need harmonization table')

    if dfs:
        file_years = [f for f in csv_files if any(str(y) in f for y in range(22, 27))]
        max_year = max([int(y) for f in file_years for y in range(22, 27) if str(y) in f], default=22)
        if max_year < 25:
            red_flags.append(f'Using FIFA {max_year} ratings — 2 seasons behind current; EA FC 25 or 26 would be fresher')

if red_flags:
    print('RED FLAGS:')
    for f in red_flags:
        print(' -', f)
else:
    print('No red flags.')

print()
print('MODELING NOTE: Position-weighted top-11 mean overall is the recommended squad feature.')
print('Use alongside Transfermarkt market values — both capture different aspects of squad quality.')